# 13 - Edge-case tests

Each of the four known issue types is tested in two ways:

1. **Real check** - a detector runs against the real pipeline tables. A healthy pipeline finds 0 issues.
2. **Control check** - the same detector runs on a copy of the data with one bad row injected on purpose. It must find that row, otherwise the detector itself is broken.

A test PASSES only when the real check finds 0 issues and the control check finds at least 1.
Nothing is written back to the real tables.

In [0]:
from pyspark.sql import functions as F

ANALYSIS_DATE = "2026-10-30"

membership_df = spark.table("segment_membership")
consent_df = spark.table("consent")
order_df = spark.table("order_master")
order_line_df = spark.table("order_line")
rfm_df = spark.table("customer_rfm")
recommendation_df = spark.table("customer_recommendation")
outcome_df = spark.table("action_outcome")
action_catalog_df = spark.table("action_catalog")

print("All tables loaded successfully!")

All tables loaded successfully!


In [0]:
# Catalogue of the issue types being tested
ground_truth_data = [
    ("ISSUE-001", "CONFLICTING_SEGMENTS",
     "One customer is assigned to two exclusive segments",
     "HIGH"),

    ("ISSUE-002", "CONSENT_VIOLATION",
     "Marketing action is assigned to a customer without granted consent",
     "CRITICAL"),

    ("ISSUE-003", "RETURN_NOT_NETTED",
     "Returned order spending is incorrectly included in monetary value",
     "HIGH"),

    ("ISSUE-004", "WRONG_ACTION_ATTRIBUTION",
     "Outcome is connected to the wrong recommended action",
     "HIGH")
]

ground_truth_df = spark.createDataFrame(
    ground_truth_data,
    ["issue_id", "issue_type", "description", "severity"]
)

ground_truth_df.write.format("delta").mode("overwrite").saveAsTable("ground_truth")

display(ground_truth_df)

issue_id,issue_type,description,severity
ISSUE-001,CONFLICTING_SEGMENTS,One customer is assigned to two exclusive segments,HIGH
ISSUE-002,CONSENT_VIOLATION,Marketing action is assigned to a customer without granted consent,CRITICAL
ISSUE-003,RETURN_NOT_NETTED,Returned order spending is incorrectly included in monetary value,HIGH
ISSUE-004,WRONG_ACTION_ATTRIBUTION,Outcome is connected to the wrong recommended action,HIGH


In [0]:
# ISSUE-001: a customer must have exactly one ACTIVE segment

def find_conflicting_segments(membership):
    return (
        membership
        .filter(F.col("membership_status") == "ACTIVE")
        .groupBy("customer_id")
        .agg(F.countDistinct("segment_id").alias("segment_count"))
        .filter(F.col("segment_count") > 1)
    )

# Real check
real_conflicts = find_conflicting_segments(membership_df).count()

# Control: copy one real membership into a different segment
injected_membership = (
    membership_df
    .limit(1)
    .withColumn("membership_id", F.concat(F.lit("TEST-"), F.col("membership_id")))
    .withColumn(
        "segment_id",
        F.when(F.col("segment_id") == "SEG-2026-000005", F.lit("SEG-2026-000006"))
         .otherwise(F.lit("SEG-2026-000005"))
    )
)

control_conflicts = find_conflicting_segments(
    membership_df.unionByName(injected_membership)
).count()

print("ISSUE-001 real conflicts:", real_conflicts, "| control detected:", control_conflicts)

ISSUE-001 real conflicts: 0 | control detected: 1


In [0]:
# ISSUE-002: a MARKETING action needs GRANTED consent in the consent table itself

def find_consent_violations(recommendations, consent):
    return (
        recommendations
        .filter(F.col("action_type") == "MARKETING")
        .join(
            consent.select("customer_id", F.col("consent_status").alias("current_consent")),
            "customer_id",
            "left"
        )
        .filter(
            F.col("current_consent").isNull() |
            (F.col("current_consent") != "GRANTED")
        )
    )

# Real check
real_consent_violations = find_consent_violations(recommendation_df, consent_df).count()

# Control: turn one recommendation of a customer with WITHDRAWN consent into a MARKETING action
injected_recommendation = (
    recommendation_df
    .join(
        consent_df.filter(F.col("consent_status") == "WITHDRAWN").select("customer_id"),
        "customer_id",
        "left_semi"
    )
    .limit(1)
    .withColumn("action_type", F.lit("MARKETING"))
)

control_consent_violations = find_consent_violations(
    recommendation_df.unionByName(injected_recommendation),
    consent_df
).count()

print("ISSUE-002 real violations:", real_consent_violations,
      "| control detected:", control_consent_violations)

ISSUE-002 real violations: 0 | control detected: 1


In [0]:
# ISSUE-003: monetary value must only include DELIVERED orders up to the analysis date
# (returned, cancelled and pending orders must not count)

def expected_monetary(orders, lines, statuses):
    order_amounts = (
        lines
        .groupBy("order_id")
        .agg(F.sum("line_total").alias("order_amount"))
    )
    return (
        orders
        .filter(
            F.col("order_status").isin(statuses) &
            (F.col("order_date") <= F.lit(ANALYSIS_DATE).cast("date"))
        )
        .join(order_amounts, "order_id", "left")
        .groupBy("customer_id")
        .agg(
            F.sum(F.coalesce(F.col("order_amount"), F.lit(0))).alias("expected_monetary")
        )
    )


def find_monetary_mismatches(rfm, expected):
    return (
        rfm.select("customer_id", "monetary")
        .join(expected, "customer_id", "left")
        .withColumn(
            "expected_monetary",
            F.coalesce(F.col("expected_monetary"), F.lit(0))
        )
        .filter(F.abs(F.col("monetary") - F.col("expected_monetary")) > 0.01)
    )

delivered_expected = expected_monetary(order_df, order_line_df, ["DELIVERED"])

# Real check: customer_rfm.monetary must equal the independent recomputation
real_return_issues = find_monetary_mismatches(rfm_df, delivered_expected).count()

# Control: an RFM table that wrongly counts RETURNED orders as spending
bad_rfm = (
    expected_monetary(order_df, order_line_df, ["DELIVERED", "RETURNED"])
    .withColumnRenamed("expected_monetary", "monetary")
)

control_return_issues = find_monetary_mismatches(bad_rfm, delivered_expected).count()

print("ISSUE-003 real mismatches:", real_return_issues,
      "| control detected:", control_return_issues)

ISSUE-003 real mismatches: 0 | control detected: 189


In [0]:
# ISSUE-004: every outcome must point to the same recommendation, action id and action name
# as the recommendation it was measured for, and to an action that exists in the catalog

def find_wrong_attribution(outcomes, recommendations, catalog):
    recommendation_actions = recommendations.select(
        "recommendation_id",
        F.col("action_id").alias("rec_action_id"),
        F.col("recommended_action").alias("rec_action_name")
    )
    catalog_ids = catalog.select(F.col("action_id").alias("catalog_action_id"))

    return (
        outcomes
        .join(recommendation_actions, "recommendation_id", "left")
        .join(catalog_ids, F.col("action_id") == F.col("catalog_action_id"), "left")
        .filter(
            F.col("rec_action_id").isNull() |
            F.col("catalog_action_id").isNull() |
            (~F.col("action_id").eqNullSafe(F.col("rec_action_id"))) |
            (~F.col("recommended_action").eqNullSafe(F.col("rec_action_name")))
        )
    )

# Real check
real_attribution_issues = find_wrong_attribution(
    outcome_df, recommendation_df, action_catalog_df
).count()

# Control: one outcome pointing at an action that does not exist
injected_outcome = (
    outcome_df
    .limit(1)
    .withColumn("outcome_id", F.concat(F.lit("TEST-"), F.col("outcome_id")))
    .withColumn("action_id", F.lit("ACTN-2026-999999"))
)

control_attribution_issues = find_wrong_attribution(
    outcome_df.unionByName(injected_outcome), recommendation_df, action_catalog_df
).count()

print("ISSUE-004 real issues:", real_attribution_issues,
      "| control detected:", control_attribution_issues)

ISSUE-004 real issues: 0 | control detected: 1


In [0]:
# Collect the results and save them (real tables are never modified)
checks = [
    ("ISSUE-001", "CONFLICTING_SEGMENTS", real_conflicts, control_conflicts),
    ("ISSUE-002", "CONSENT_VIOLATION", real_consent_violations, control_consent_violations),
    ("ISSUE-003", "RETURN_NOT_NETTED", real_return_issues, control_return_issues),
    ("ISSUE-004", "WRONG_ACTION_ATTRIBUTION", real_attribution_issues, control_attribution_issues),
]

def status_of(real_issues, control_detected):
    if control_detected == 0:
        return "DETECTOR_BROKEN"   # the test could not catch an injected problem
    if real_issues > 0:
        return "ISSUES_FOUND"      # the real pipeline output has this problem
    return "PASS"

result_rows = [
    (issue_id, issue_type, int(real), int(control), status_of(real, control))
    for issue_id, issue_type, real, control in checks
]

results_df = (
    spark.createDataFrame(
        result_rows,
        ["issue_id", "issue_type", "real_issues_found", "injected_issues_detected", "status"]
    )
    .withColumn("test_date", F.current_date())
)

results_df.write.format("delta").mode("overwrite").saveAsTable("edge_case_test_results")

display(results_df)

issue_id,issue_type,real_issues_found,injected_issues_detected,status,test_date
ISSUE-001,CONFLICTING_SEGMENTS,0,1,PASS,2026-11-08
ISSUE-002,CONSENT_VIOLATION,0,1,PASS,2026-11-08
ISSUE-003,RETURN_NOT_NETTED,0,189,PASS,2026-11-08
ISSUE-004,WRONG_ACTION_ATTRIBUTION,0,1,PASS,2026-11-08


In [0]:
# Summary
not_passed = [row for row in result_rows if row[4] != "PASS"]

for issue_id, issue_type, real, control, status in result_rows:
    print(f"{issue_id}  {issue_type:26s} real={real:<5} control={control:<5} {status}")

if not_passed:
    print(f"\n{len(not_passed)} of {len(result_rows)} edge-case tests did not pass - see edge_case_test_results")
else:
    print("\nAll edge-case tests passed")

ISSUE-001  CONFLICTING_SEGMENTS       real=0     control=1     PASS
ISSUE-002  CONSENT_VIOLATION          real=0     control=1     PASS
ISSUE-003  RETURN_NOT_NETTED          real=0     control=189   PASS
ISSUE-004  WRONG_ACTION_ATTRIBUTION   real=0     control=1     PASS

All edge-case tests passed
